# Low-ℓ BB — Source Patch Inspector

Finds rough point-source positions from the mask, then plots 3°×3° patches of the raw map centred on each source.

1. Load T map + build observed-pixel mask
2. Load apodisation mask → downgrade to nside=64 → threshold → source RA/Dec list
3. Choose a source by index, plot, close, repeat

In [ ]:
import os
import sys
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

In [ ]:
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, _here)

from Plot import apply_spt_style, show_map_thumbnail
apply_spt_style()

#### Paths

In [ ]:
# Map — T only, used for obs mask and background image
DATA_DIR   = "/sptgrid/analysis/spt3g_d1_midell_tqu_healpix"
COADD_FILE = os.path.join(DATA_DIR, "real_data_maps", "full", "full_220ghz.fits")

# Mask — with disk masking so source holes are clear
MASK_DIR  = "/sptlocal/user/creichardt/bb2020"
MASK_FILE = "puremask8192_0p5medwt_250mJy_30arcmin.npz"

# Source-finding parameters
NSIDE_FIND = 64      # downgrade resolution — each 30 arcmin source collapses to ~1 pixel
APOD_THRESH = 0.5    # pixels below this (within the field) are flagged as source candidates
MARGIN_DEG  = 2.0    # interior margin in degrees — excludes field-boundary low-apod pixels

# Source patch display
PATCH_DEG  = 3.0
RESO_ARCMIN = 0.5
PATCH_PIX  = int(PATCH_DEG * 60 / RESO_ARCMIN)   # 360 px
CMAP       = "coolwarm"

print(f"Map file  : {os.path.basename(COADD_FILE)}")
print(f"Mask file : {MASK_FILE}")
print(f"Patch     : {PATCH_DEG}° × {PATCH_DEG}°  =  {PATCH_PIX} × {PATCH_PIX} px  at {RESO_ARCMIN} arcmin/px")

#### Load T map

In [ ]:
# Load T only — used for the observed-pixel mask and for the source patch images
T_arr = hp.read_map(COADD_FILE, field=0, partial=False)
nside = hp.get_nside(T_arr)

obs_mask = np.isfinite(T_arr) & (T_arr != hp.UNSEEN)

# Field extent — needed for the interior margin filter
obs_pix        = np.where(obs_mask)[0]
theta_c, phi_c = hp.pix2ang(nside, obs_pix)
ra_obs  = np.degrees(phi_c)
ra_obs  = np.where(ra_obs > 180, ra_obs - 360, ra_obs)
dec_obs = 90.0 - np.degrees(theta_c)
ra_min,  ra_max  = ra_obs.min(),  ra_obs.max()
dec_min, dec_max = dec_obs.min(), dec_obs.max()

print(f"nside          : {nside}")
print(f"Observed pixels: {obs_mask.sum():,}")
print(f"RA  range      : {ra_min:.1f}° → {ra_max:.1f}°")
print(f"Dec range      : {dec_min:.1f}° → {dec_max:.1f}°")

#### Find source positions from mask

In [ ]:
# Load mask at full nside
mask_path = os.path.join(MASK_DIR, MASK_FILE)
with np.load(mask_path) as d:
    apod = d[d.files[0]].astype(float)

# Downgrade both mask and obs_mask to low resolution
# At nside=64 (~55 arcmin pixels) each 30 arcmin source hole collapses to ~1 pixel
apod_lo = hp.ud_grade(apod, NSIDE_FIND)
obs_lo  = hp.ud_grade(obs_mask.astype(float), NSIDE_FIND)

# Candidate pixels: low apod value, inside observed field
src_pix        = np.where((apod_lo < APOD_THRESH) & (obs_lo > 0.5))[0]
theta_s, phi_s = hp.pix2ang(NSIDE_FIND, src_pix)
ra_s  = np.degrees(phi_s)
ra_s  = np.where(ra_s > 180, ra_s - 360, ra_s)
dec_s = 90.0 - np.degrees(theta_s)

# Interior filter — remove field-boundary pixels
interior = (
    (ra_s  > ra_min  + MARGIN_DEG) & (ra_s  < ra_max  - MARGIN_DEG) &
    (dec_s > dec_min + MARGIN_DEG) & (dec_s < dec_max - MARGIN_DEG)
)
src_ra  = ra_s[interior]
src_dec = dec_s[interior]

print(f"Mask loaded : {MASK_FILE}")
print(f"Sources found (rough): {len(src_ra)}")
print()
print(f"{'IDX':>4}  {'RA':>8}  {'Dec':>8}")
for i, (r, d) in enumerate(zip(src_ra, src_dec)):
    print(f"{i:4d}  {r:+8.2f}°  {d:+8.2f}°")

#### Source patch viewer

Set `SOURCE_IDX` to a number from the list above, run the plot cell, inspect, close, repeat.

In [ ]:
# Choose source
SOURCE_IDX = 0

ra_c  = src_ra[SOURCE_IDX]
dec_c = src_dec[SOURCE_IDX]
print(f"Source {SOURCE_IDX}  →  RA {ra_c:+.2f}°  Dec {dec_c:+.2f}°")

In [ ]:
# Plot unmasked T around the source
rms        = float(np.std(T_arr[obs_mask]))
vmin, vmax = -3 * rms, 3 * rms   # wider range to show bright sources

show_map_thumbnail(
    T_arr,
    vmin=vmin, vmax=vmax,
    title=f"T  unmasked  |  source {SOURCE_IDX}  |  RA {ra_c:+.2f}°  Dec {dec_c:+.2f}°",
    unit="Tcmb", cmap=CMAP,
    rot=(ra_c, dec_c, 0),
    xsize=PATCH_PIX, ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
plt.show()

In [ ]:
# Close
plt.close("all")